# Notebook 09 — Variational Inference: Approximate Posteriors at Scale

## Background

MCMC (NUTS) gives you exact posterior samples — in principle. In practice it has two limitations: (1) it requires running multiple chains for many iterations, which is slow for large models or large datasets, and (2) it doesn't scale to truly high-dimensional models (like Bayesian neural networks with millions of parameters).

Variational Inference (VI) takes a different approach: instead of sampling from the true posterior `P(theta|data)`, it **approximates** the posterior with a simpler distribution `Q(theta; phi)` and optimizes `phi` so that `Q` is as close to `P` as possible. The approximation is biased — you don't get the exact posterior — but it's often much faster, and for large models it's the only practical option.

The closeness measure between `Q` and `P` is the **KL divergence** `KL(Q || P)`. Minimizing KL(Q||P) is equivalent to maximizing the **Evidence Lower Bound (ELBO)**:

```
ELBO(phi) = E_Q[log P(data|theta)] - KL(Q(theta;phi) || P(theta))
           = E_Q[log P(theta, data)] - E_Q[log Q(theta; phi)]
```

The ELBO lower-bounds the log marginal likelihood `log P(data)`. Maximizing it simultaneously minimizes the KL divergence and increases the evidence.

## What You'll Learn

- The ELBO objective: what we're optimizing and why
- Mean-field VI: the fully factored approximation `Q = prod Q_i(theta_i)`
- PyMC's ADVI: Automatic Differentiation Variational Inference
- When VI works well and when it fails (posterior correlations, multimodality)
- ADVI vs MCMC: speed/accuracy tradeoffs
- Full-rank VI: beyond mean-field

## Why This Matters

ADVI is how you get Bayesian inference at scale. It powers large topic models, Bayesian neural networks, and variational autoencoders. In PyMC, `pm.fit()` runs ADVI in seconds where `pm.sample()` might take minutes — though with a cost in accuracy. Understanding that tradeoff lets you choose the right tool.

In [ ]:
import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

print(f'PyMC {pm.__version__}  |  ArviZ {az.__version__}')

## Part 1 — The ELBO: What We're Optimizing

The ELBO comes from decomposing the log marginal likelihood:

```
log P(data) = ELBO(Q) + KL(Q || P_posterior)
```

Since `KL >= 0`, ELBO is always a lower bound on `log P(data)`. Maximizing ELBO over `Q`:
1. Pushes ELBO closer to `log P(data)` (better model evidence)
2. Forces `KL(Q || P_posterior)` toward zero (Q approaches the true posterior)

The ELBO can be rewritten as:

```
ELBO = E_Q[log P(data|theta)] - KL(Q || prior)
```

- First term: **expected log-likelihood** — how well the model explains the data
- Second term: **KL regularization** — how much `Q` deviates from the prior

This is exactly the same structure as regularized maximum likelihood, but with distributions instead of point estimates.

In [ ]:
# Visualize: approximating a non-Gaussian posterior with a Gaussian Q
# This illustrates what mean-field VI does (and where it fails)

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# True posterior: bimodal (mixture of two Gaussians)
def log_target_bimodal(theta):
    """Log of a bimodal distribution: illustrates where mean-field VI fails."""
    log_p1 = stats.norm.logpdf(theta, -2, 0.8)
    log_p2 = stats.norm.logpdf(theta,  2, 0.8)
    # log-sum-exp: log(0.5 * p1 + 0.5 * p2)
    return np.log(0.5 * np.exp(log_p1) + 0.5 * np.exp(log_p2))

def log_target_correlated(theta1, theta2, rho=0.9):
    """Log of a highly correlated bivariate Gaussian.
    Mean-field VI (independent Q) can't capture the correlation.
    """
    cov_inv = np.array([[1, -rho], [-rho, 1]]) / (1 - rho**2)
    theta = np.stack([theta1, theta2], axis=-1)
    return -0.5 * np.einsum('...i,ij,...j', theta, cov_inv, theta)

# 1D case: mean-field VI approximating a bimodal posterior
theta_range = np.linspace(-6, 6, 1000)
true_log_p = log_target_bimodal(theta_range)
true_p = np.exp(true_log_p - true_log_p.max())  # normalize for visualization
true_p /= np.trapz(true_p, theta_range)

# Optimal Gaussian Q: minimize KL(Q || P_true) = find the Gaussian closest to the bimodal
# The optimal Q has mean 0 (by symmetry) and variance that minimizes KL
# For mean-field VI with Gaussian Q on a bimodal target:
# KL(Q||P) favors Q that is broader (zero-avoiding behavior)
# For KL(P||Q) (expectation propagation): Q would be too narrow

best_sigma = 2.5  # approximate optimal sigma for KL(Q||P)
q_vi = stats.norm.pdf(theta_range, 0, best_sigma)

# KL(P||Q) -- expectation propagation direction
q_ep_sigma = 1.2
q_ep = stats.norm.pdf(theta_range, 0, q_ep_sigma)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(theta_range, true_p, 'black', lw=2, label='True posterior (bimodal)')
ax.plot(theta_range, q_vi / np.trapz(q_vi, theta_range),
        'steelblue', lw=2, linestyle='--',
        label=f'VI (KL(Q||P)): N(0, {best_sigma}^2)\n  "mass-covering" -- too broad')
ax.plot(theta_range, q_ep / np.trapz(q_ep, theta_range),
        'darkorange', lw=2, linestyle='-.',
        label=f'EP (KL(P||Q)): N(0, {q_ep_sigma}^2)\n  "mode-seeking" -- too narrow')
ax.set_xlabel('theta'); ax.set_ylabel('Density')
ax.set_title('Gaussian Approximation of Bimodal Posterior\nVI is mass-covering; EP is mode-seeking')
ax.legend(fontsize=8)

# 2D correlated case: mean-field Q can't capture correlation
ax = axes[1]
rho_true = 0.9
t1 = np.linspace(-4, 4, 200)
T1, T2 = np.meshgrid(t1, t1)
P_true_2d = np.exp(log_target_correlated(T1, T2, rho=rho_true))
Q_mf_2d   = stats.norm.pdf(T1, 0, 1) * stats.norm.pdf(T2, 0, 1)  # independent

ax.contour(T1, T2, P_true_2d, levels=5, colors='black', linewidths=2, linestyles='-')
ax.contour(T1, T2, Q_mf_2d, levels=5, colors='steelblue', linewidths=2, linestyles='--')
ax.set_xlabel('theta_1'); ax.set_ylabel('theta_2')
ax.set_title(f'Mean-field VI on Correlated Posterior (rho={rho_true})\n'
             f'Black=true, Blue=mean-field Q (ignores correlation)')

plt.suptitle('Variational Inference Failure Modes', fontsize=12)
plt.tight_layout()
plt.show()

print('Mean-field VI limitations:')
print('  1. Cannot represent multimodal posteriors (picks one mode)')
print('  2. Cannot represent correlated parameters (factored Q)')
print('  3. Tends to underestimate posterior variance (mean-seeking)')

## Part 2 — ADVI in PyMC

PyMC implements Automatic Differentiation Variational Inference (Kucukelbir et al. 2017). ADVI:
1. Transforms all constrained parameters to unconstrained space (log for positive, logit for [0,1], etc.)
2. Fits a Gaussian `Q` in the unconstrained space using stochastic gradient ascent on the ELBO
3. For mean-field: fits a diagonal Gaussian (independent dimensions)
4. For full-rank: fits a dense Gaussian (can capture correlations; `O(d^2)` parameters)

In PyMC, `pm.fit()` runs ADVI.

In [ ]:
# Compare ADVI vs NUTS on a simple logistic regression
import time

np.random.seed(42)
n_vi = 300
x_vi = np.random.normal(0, 1, n_vi)
y_vi = np.random.binomial(1, 1/(1+np.exp(-(1.5*x_vi - 0.5))))

with pm.Model() as logistic_vi:
    alpha = pm.Normal('alpha', mu=0, sigma=3)
    beta  = pm.Normal('beta',  mu=0, sigma=3)
    y_obs = pm.Bernoulli('y_obs', logit_p=alpha + beta*x_vi, observed=y_vi)
    
    # ADVI (mean-field)
    t0_advi = time.perf_counter()
    approx_mf = pm.fit(
        n=20_000,                          # ELBO optimization steps
        method='advi',                     # mean-field VI
        progressbar=True,
        random_seed=42
    )
    t_advi = time.perf_counter() - t0_advi
    
    # Sample from the VI posterior
    vi_trace = approx_mf.sample(2000)

print(f'\nADVI time: {t_advi:.2f}s')

with logistic_vi:
    t0_nuts = time.perf_counter()
    nuts_trace = pm.sample(draws=2000, tune=1000, chains=4,
                            random_seed=42, progressbar=False)
    t_nuts = time.perf_counter() - t0_nuts

print(f'NUTS time: {t_nuts:.2f}s')
print(f'Speedup: {t_nuts/t_advi:.1f}x faster with ADVI')

In [ ]:
# Compare posteriors: ADVI vs NUTS
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, param, true_val in zip(axes,
                                ['alpha', 'beta'],
                                [-0.5,    1.5]):
    vi_samp   = vi_trace.posterior[param].values.flatten()
    nuts_samp = nuts_trace.posterior[param].values.flatten()
    
    xr = np.linspace(min(vi_samp.min(), nuts_samp.min()) - 0.5,
                     max(vi_samp.max(), nuts_samp.max()) + 0.5, 300)
    
    kde_vi   = stats.gaussian_kde(vi_samp)
    kde_nuts = stats.gaussian_kde(nuts_samp)
    
    ax.fill_between(xr, kde_vi(xr), alpha=0.25, color='steelblue')
    ax.plot(xr, kde_vi(xr), 'steelblue', lw=2, label='ADVI (mean-field)')
    ax.fill_between(xr, kde_nuts(xr), alpha=0.15, color='darkorange')
    ax.plot(xr, kde_nuts(xr), 'darkorange', lw=2, label='NUTS (exact)')
    ax.axvline(true_val, color='red', linestyle='--', lw=1.5, label=f'True: {true_val}')
    ax.set_xlabel(param); ax.set_ylabel('Density')
    ax.set_title(f'Posterior of {param}: ADVI vs NUTS')
    ax.legend(fontsize=9)

plt.suptitle('ADVI vs NUTS: Posterior Comparison (Logistic Regression)', fontsize=12)
plt.tight_layout()
plt.show()

# Summary statistics comparison
print(f'{"":>10} {"ADVI mean":>12} {"ADVI std":>10} {"NUTS mean":>12} {"NUTS std":>10}')
print('-' * 58)
for param, true_val in zip(['alpha', 'beta'], [-0.5, 1.5]):
    vi_s   = vi_trace.posterior[param].values.flatten()
    nuts_s = nuts_trace.posterior[param].values.flatten()
    print(f'{param:>10} {vi_s.mean():>12.3f} {vi_s.std():>10.3f} {nuts_s.mean():>12.3f} {nuts_s.std():>10.3f}  (true={true_val})')

In [ ]:
# ELBO convergence plot -- monitor to check if VI converged
fig, ax = plt.subplots(figsize=(10, 4))

elbo_values = -approx_mf.hist['loss']  # PyMC stores -ELBO as 'loss'
# Smooth with rolling mean
window = 200
elbo_smooth = np.convolve(elbo_values, np.ones(window)/window, mode='valid')

ax.plot(elbo_values[::10], alpha=0.3, color='steelblue', lw=0.5, label='Raw ELBO')
ax.plot(np.arange(len(elbo_smooth))*10, elbo_smooth, 'steelblue', lw=2, label=f'Smoothed (window={window})')
ax.set_xlabel('Optimization step'); ax.set_ylabel('ELBO')
ax.set_title('ELBO Convergence During ADVI\nFlattening = convergence; still declining = needs more steps')
ax.legend()
plt.tight_layout()
plt.show()

print('ELBO diagnostics:')
print(f'  Initial ELBO: {elbo_values[:100].mean():.1f}')
print(f'  Final ELBO:   {elbo_values[-100:].mean():.1f}')
print(f'  Converged?    {"Yes" if elbo_values[-200:].std() < abs(elbo_values[-200:].mean()) * 0.01 else "Not sure -- try more steps"}')

## Part 3 — Full-Rank VI

Mean-field VI assumes all parameters are independent in `Q` — it fits a diagonal covariance Gaussian. Full-rank VI fits a dense covariance matrix, which can capture posterior correlations at the cost of `O(d^2)` parameters. For moderate-dimensional models (d < a few hundred), full-rank VI is much more accurate.

In [ ]:
# Full-rank VI on the same logistic regression model
with logistic_vi:
    t0_fr = time.perf_counter()
    approx_fr = pm.fit(
        n=20_000,
        method='fullrank_advi',  # dense covariance matrix
        progressbar=True,
        random_seed=42
    )
    t_fr = time.perf_counter() - t0_fr
    vi_trace_fr = approx_fr.sample(2000)

print(f'Full-rank ADVI time: {t_fr:.2f}s')

# Compare mean-field vs full-rank vs NUTS
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, param, true_val in zip(axes, ['alpha', 'beta'], [-0.5, 1.5]):
    vi_mf_s = vi_trace.posterior[param].values.flatten()
    vi_fr_s = vi_trace_fr.posterior[param].values.flatten()
    nuts_s  = nuts_trace.posterior[param].values.flatten()
    
    xr = np.linspace(
        min(vi_mf_s.min(), nuts_s.min()) - 0.5,
        max(vi_mf_s.max(), nuts_s.max()) + 0.5, 300
    )
    ax.plot(xr, stats.gaussian_kde(vi_mf_s)(xr), 'steelblue', lw=2,  label='ADVI (mean-field)')
    ax.plot(xr, stats.gaussian_kde(vi_fr_s)(xr), 'seagreen',  lw=2,  label='ADVI (full-rank)')
    ax.plot(xr, stats.gaussian_kde(nuts_s)(xr),  'darkorange', lw=2, label='NUTS (exact)')
    ax.axvline(true_val, color='red', linestyle='--', lw=1.5)
    ax.set_xlabel(param); ax.set_title(f'{param}')
    ax.legend(fontsize=9)

plt.suptitle('Mean-field vs Full-rank ADVI vs NUTS', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate: ADVI fails on a bimodal posterior
# This is the fundamental limitation -- ADVI finds one mode

np.random.seed(42)
n_bimodal = 200

# True generating process: mixture of two regimes
# Which regime each observation is from is unknown
x_bm = np.random.uniform(-3, 3, n_bimodal)
# Some observations from regime 1 (beta~+2), some from regime 2 (beta~-2)
regime = np.random.binomial(1, 0.5, n_bimodal)
y_bm = np.where(regime == 1,
                2*x_bm + np.random.normal(0, 0.5, n_bimodal),
                -2*x_bm + np.random.normal(0, 0.5, n_bimodal))

with pm.Model() as bimodal_model:
    beta = pm.Normal('beta', mu=0, sigma=5)
    sigma = pm.HalfNormal('sigma', sigma=2)
    y_obs = pm.Normal('y_obs', mu=beta * x_bm, sigma=sigma, observed=y_bm)
    
    # ADVI
    approx_bm = pm.fit(n=20_000, method='advi', random_seed=42, progressbar=False)
    vi_bm_trace = approx_bm.sample(2000)
    
    # NUTS
    nuts_bm_trace = pm.sample(draws=2000, tune=1000, chains=4,
                               random_seed=42, progressbar=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

vi_beta_samp   = vi_bm_trace.posterior['beta'].values.flatten()
nuts_beta_samp = nuts_bm_trace.posterior['beta'].values.flatten()

for ax, (samp, label, color) in zip(axes, [
    (vi_beta_samp,   'ADVI (mean-field)', 'steelblue'),
    (nuts_beta_samp, 'NUTS (exact)',       'darkorange'),
]):
    ax.hist(samp, bins=60, density=True, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(2.0,  color='green', linestyle='--', lw=1.5, label='True beta=+2')
    ax.axvline(-2.0, color='red',   linestyle='--', lw=1.5, label='True beta=-2')
    ax.set_xlabel('beta'); ax.set_title(label)
    ax.legend(fontsize=9)

plt.suptitle('ADVI on Bimodal Posterior -- ADVI Picks One Mode, NUTS Finds Both', fontsize=12)
plt.tight_layout()
plt.show()

print('ADVI fundamental limitation: Gaussian Q cannot represent multiple modes.')
print('When the posterior is multimodal, ADVI will find one mode and miss the rest.')
print('NUTS (given enough warmup) finds all modes.')

## Part 4 — When to Use VI vs MCMC

The tradeoffs are clear once you understand what VI sacrifices:

In [ ]:
print('ADVI vs NUTS: Decision Guide')
print('=' * 65)
print()
comparisons = [
    ('Speed',         'Fast (seconds-minutes)', 'Slow (minutes-hours)'),
    ('Accuracy',      'Biased (Gaussian Q)',    'Exact (given convergence)'),
    ('Multimodal',    'Fails (picks one mode)', 'Works (finds all modes)'),
    ('Correlations',  'MF: misses; FR: ok',     'Full posterior'),
    ('Scalability',   'O(d) params (MF)',       'O(n) gradient evals'),
    ('Diagnostics',   'ELBO convergence',       'R-hat, ESS, divergences'),
    ('BNN support',   'Yes (natural fit)',      'Struggles at high-d'),
]
print(f'{"Property":20} {"ADVI":28} {"NUTS":28}')
print('-' * 78)
for prop, advi, nuts in comparisons:
    print(f'{prop:20} {advi:28} {nuts:28}')

print()
print('Use ADVI when:')
print('  - n > 50,000 (large data: NUTS per-sample cost too high)')
print('  - d > 1000 parameters (NUTS scales poorly in high dimensions)')
print('  - Posterior is roughly Gaussian/unimodal')
print('  - You need quick exploratory inference (then validate with NUTS)')
print()
print('Use NUTS when:')
print('  - Accuracy matters (publishable research, safety-critical decisions)')
print('  - Posterior may be multimodal')
print('  - Strong parameter correlations (hierarchical models)')
print('  - n < 50,000 and d < a few hundred')

In [ ]:
# ADVI for quick exploration, NUTS for final inference -- a practical workflow

np.random.seed(42)
n_big = 5000   # larger dataset
x_big = np.random.normal(0, 1, n_big)
y_big = np.random.binomial(1, 1/(1+np.exp(-(0.8*x_big + 0.3))))

with pm.Model() as big_model:
    alpha = pm.Normal('alpha', mu=0, sigma=3)
    beta  = pm.Normal('beta',  mu=0, sigma=3)
    y_obs = pm.Bernoulli('y_obs', logit_p=alpha + beta*x_big, observed=y_big)
    
    print('Step 1: Quick ADVI for initial exploration...')
    t0 = time.perf_counter()
    approx_big = pm.fit(n=10_000, method='advi', progressbar=False, random_seed=42)
    vi_big = approx_big.sample(1000)
    print(f'  Done in {time.perf_counter()-t0:.1f}s')
    print(f'  ADVI alpha: {vi_big.posterior["alpha"].values.mean():.3f}')
    print(f'  ADVI beta:  {vi_big.posterior["beta"].values.mean():.3f}')
    print()
    
    print('Step 2: NUTS for final validated inference...')
    t0 = time.perf_counter()
    nuts_big = pm.sample(draws=1000, tune=500, chains=4,
                          random_seed=42, progressbar=False)
    print(f'  Done in {time.perf_counter()-t0:.1f}s')
    print(f'  NUTS alpha: {nuts_big.posterior["alpha"].values.mean():.3f}')
    print(f'  NUTS beta:  {nuts_big.posterior["beta"].values.mean():.3f}')

print()
print('True: alpha=0.3, beta=0.8')

## Summary

**Variational Inference** approximates the posterior by optimizing a simpler distribution `Q` to maximize the ELBO. PyMC's `pm.fit()` implements ADVI:

```python
with pm.Model() as model:
    # ... define model ...
    
    # Mean-field ADVI (fast, independent dimensions)
    approx = pm.fit(n=20_000, method='advi')
    trace_vi = approx.sample(2000)
    
    # Full-rank ADVI (slower, captures correlations)
    approx_fr = pm.fit(n=20_000, method='fullrank_advi')
```

**ELBO convergence check**: `approx.hist['loss']` gives the optimization history. If still declining after `n` steps, increase `n`.

**Limitations**:
- Mean-field: misses posterior correlations (use full-rank for correlated models)
- Both: cannot represent multimodal posteriors (fundamental Gaussian Q limitation)
- Both: variance is generally underestimated compared to true posterior

**Bottom line**: ADVI is the right tool when NUTS is too slow. For high-stakes inference, validate ADVI results with NUTS on a smaller dataset first.

**Next**: Notebook 10 — Bayesian Neural Networks. Weight uncertainty, MC dropout, and why Bayesian deep learning matters for calibration.